# Aster Payroll Gym — explicit Colab GPU execution

Run this notebook in **Google Colab with a GPU**. Do not run it on your laptop. The default repository install does not contain training packages. No results in this notebook are prefilled. Follow `docs/colab.md`; preserve failure evidence.


In [ ]:
import importlib.util
import os
import platform
from pathlib import Path

try:
    found = importlib.util.find_spec("google.colab") is not None
except (ModuleNotFoundError, ValueError):
    found = False
COLAB_ONLY = platform.system() == "Linux" and Path("/content").is_dir() and found and any(os.environ.get(k) for k in ("COLAB_RELEASE_TAG", "COLAB_BACKEND_VERSION"))
if not COLAB_ONLY:
    raise RuntimeError("STOP: this notebook is for hosted Google Colab only. No local fallback.")
print("Recognized hosted Colab. Select a GPU before proceeding.")


## Clone and install in Colab
Replace the URL with your repository. This cell changes only the Colab filesystem. Private repositories can be uploaded/cloned using your preferred authenticated Colab workflow; never paste credentials into committed cells.


In [ ]:
assert COLAB_ONLY
import subprocess  # noqa: E402
import sys  # noqa: E402

REPO_URL = "https://github.com/sahajrajmalla/aster-payroll-gym.git"
if "YOUR_ACCOUNT" in REPO_URL:
    raise ValueError("Replace REPO_URL with your repository URL")
repo = Path("/content/aster-payroll-gym")
if not repo.exists():
    subprocess.run(["git", "clone", REPO_URL, str(repo)], check=True)
os.chdir(repo)
subprocess.run([sys.executable, "-m", "pip", "install", "uv==0.12.19"], check=True)
subprocess.run(["uv", "sync", "--frozen", "--extra", "cloud", "--no-dev"], check=True)


## Persistent cloud storage, credentials, and adjustable configuration
Mount Drive for checkpoints. Add `JUDGE_API_KEY` using the Colab Secrets panel. Verify current free-tier eligibility and model availability before setting the free-tier flag. Never enable billing for this workflow.


In [ ]:
assert COLAB_ONLY
from google.colab import drive, userdata  # noqa: E402

drive.mount("/content/drive")
os.environ["JUDGE_API_KEY"] = userdata.get("JUDGE_API_KEY")
os.environ["JUDGE_MODEL"] = "gemini-3.5-flash-lite"  # Verify availability on your account.
FREE_TIER_CONFIRMED = False  # Set True only after verifying your account and disabling billing.
if not FREE_TIER_CONFIRMED:
    raise RuntimeError("Confirm zero-dollar judge access before continuing.")
os.environ["JUDGE_VERIFIED_FREE"] = "true"
os.environ["JUDGE_MAX_CALLS"] = "100"  # A bounded starting quota, not an implied paid allowance.
import json  # noqa: E402

config = json.loads(Path("configs/train.json").read_text())
config["output_dir"] = "/content/drive/MyDrive/aster-gym-results"
# To reduce compute, use 40 for BOTH sweeps and choose a fresh output_dir.
# config["steps"] = 40
Path("/content/aster-train.json").write_text(json.dumps(config, indent=2))
print({k: config[k] for k in ("output_dir", "steps", "batch_size", "gradient_accumulation_steps", "beta_values")})


In [ ]:
assert COLAB_ONLY
import shutil  # noqa: E402

if shutil.disk_usage('/content').free < 5 * 1024**3:
    raise RuntimeError('At least 5 GiB of cloud filesystem space is required.')
probe_path = Path(config['output_dir']) / '.write-test'
probe_path.parent.mkdir(parents=True, exist_ok=True)
probe_path.write_text('cloud-storage-test')
probe_path.unlink()
UV = ['uv', 'run', '--frozen', '--extra', 'cloud', '--no-dev']
preflight = """from importlib.metadata import version
from aster_gym.cloud.guard import require_cuda
t = require_cuda(explicit=True)
for package, expected in {'torch':'2.8.0','transformers':'4.57.1','trl':'0.26.2','verifiers':'0.1.14','peft':'0.17.1','accelerate':'1.10.1'}.items():
    assert version(package).split('+')[0] == expected, (package, version(package))
import verifiers as vf
from aster_gym.environment import load_environment
for mode in ('single', 'tool'):
    assert isinstance(load_environment(mode=mode, n=3, judge=None), vf.Environment)
print({'cuda_device':t.cuda.get_device_name(0),'optional_environment_startup':'passed'})
"""
subprocess.run(UV + ['python', '-c', preflight], check=True)


## Guarded preflight and real three-step smoke
The package repeats the Colab check before importing PyTorch and refuses non-CUDA execution. Run the smoke explicitly, inspect its evidence, and fix cloud errors there. A zero-variance run may make no update; report this result honestly.


In [ ]:
assert COLAB_ONLY
START_SMOKE = False  # Explicitly set True after GPU/quota checks.
if not START_SMOKE:
    raise RuntimeError("Set START_SMOKE=True to start the cloud smoke run.")
UV = ["uv", "run", "--frozen", "--extra", "cloud", "--no-dev"]
subprocess.run(UV + ["python", "-c", "from aster_gym.cloud.guard import require_cuda; t = require_cuda(explicit=True); print(t.cuda.get_device_name(0))"], check=True)
subprocess.run(UV + ["python", "-m", "aster_gym.cloud.train", "--config", "/content/aster-train.json", "--start-training", "--smoke"], check=True)


In [ ]:
assert COLAB_ONLY
smoke = Path(config["output_dir"]) / "smoke"
for filename in ("config.json", "reference_proof.json"):
    print(filename, json.loads((smoke / filename).read_text()))
print((smoke / "training_metrics.jsonl").read_text())


## Explicitly run both KL sweeps and held-out evaluation
This is the long-running GPU step. The two runs start from the same initial model. Validation chooses beta, then the initial and selected models are evaluated on the held-out seed-separated split, three rollouts per task. Checkpoints stay on Drive. To resume an interrupted run, set `RESUME = True`; do not change its config.


In [ ]:
assert COLAB_ONLY
START_SWEEPS = False  # Explicitly set True after reviewing the smoke evidence.
if not START_SWEEPS:
    raise RuntimeError("Set START_SWEEPS=True to start both cloud training sweeps.")
RESUME = False
command = UV + ["python", "-m", "aster_gym.cloud.train", "--config", "/content/aster-train.json", "--start-training"]
if RESUME:
    command.append("--resume")
subprocess.run(command, check=True)


## Additional common-cohort / transfer inference
Run only after the frozen task files exist. These lightweight orchestration cells still execute all small-model inference on Colab CUDA. Set the exact cohort path produced by the repository. Do not alter the sealed transfer tasks after observing model outputs.


In [ ]:
assert COLAB_ONLY
START_INFERENCE = False  # Explicitly start these cloud-only comparison runs.
if not START_INFERENCE:
    raise RuntimeError("Set START_INFERENCE=True to run the small-model comparisons.")
subprocess.run(UV + ["python", "-c", "from aster_gym.generator import read_taskset,write_taskset; write_taskset(read_taskset('data/evaluation.jsonl')[:30], 'data/comparison.jsonl'); write_taskset(read_taskset('data/evaluation.jsonl')[:12], 'data/tool-comparison.jsonl')"], check=True)
for tasks, name, mode in [("data/comparison.jsonl", "comparison-small", "single"), ("data/tool-comparison.jsonl", "tool-small", "tool")]:
    subprocess.run(UV + ["python", "-m", "aster_gym.cloud.evaluate", "--config", "/content/aster-train.json", "--tasks", tasks, "--output", str(Path(config["output_dir"]) / name), "--mode", mode, "--start-inference"], check=True)
if Path("reviews/transfer-freeze.json").exists():
    subprocess.run(UV + ["python", "-m", "aster_gym.cloud.evaluate", "--config", "/content/aster-train.json", "--tasks", "data/transfer.jsonl", "--output", str(Path(config["output_dir"]) / "transfer-small"), "--start-inference"], check=True)
else:
    print("Transfer pending: human approval and matching freeze marker required. Other results can still be exported.")


## Export compact results only
This export never includes model weights, optimizer state, code, or pickle. Failed/partial runs can be preserved too; they remain explicitly marked. Download this ZIP to your laptop and use `aster-gym import-results`. Keep the notebook link and full cloud checkpoints for review.


In [ ]:
assert COLAB_ONLY
subprocess.run(UV + ["python", "-c", "from aster_gym.bundles import export_bundle; import sys; print(export_bundle(sys.argv[1], sys.argv[2]))", config["output_dir"], "/content/aster-results.zip"], check=True)
from google.colab import files  # noqa: E402

files.download("/content/aster-results.zip")
